🛰️ MintPy Daily-Use Code Repository — OpenSARLab Edition
Kernel Requirement: Every cell in this notebook must use the osl_mintpy kernel.
Check and switch it in the top-right corner of your Jupyter interface.
The default Python 3 kernel will fail with ModuleNotFoundError.

Universal Path Setup — Run This First in Every Session

**What it does:** Declares a master `DATA_DIR` variable pointing to your MintPy output folder.  
All 18 snippets below depend on this variable. Run this cell first, every time.

In [1]:
import os

# Change this to your actual MintPy geo-coded output folder in OpenSARLab.
# Typical OpenSARLab paths begin with /home/jovyan/
DATA_DIR = "/home/jovyan/dhaka/HyP3_downloads/MintPy"

# Verify the path actually exists before proceeding
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_DIR}\nCheck your path and try again.")

print(f"✅ Data directory confirmed: {DATA_DIR}")

✅ Data directory confirmed: /home/jovyan/dhaka/HyP3_downloads/MintPy


16 — Mask by Connected Component (Remove Pixel Islands)

**What it does:** Loads MintPy's connected component mask — which records which pixels
belong to a single coherent, spatially connected region — and uses it to remove
isolated pixel clusters ("islands") that survived the coherence filter but are unreliable.  
**When to use:** Always apply this *after* the coherence mask (Snippet 13) for any
publication-quality result. Particularly important in urban SAR work where bright
point scatterers can create isolated high-coherence islands that are spatially
disconnected from the main network.

In [2]:
import numpy as np
from mintpy.utils import readfile

vel_path        = f"{DATA_DIR}/velocity.h5"
coh_path        = f"{DATA_DIR}/temporalCoherence.h5"
mask_path       = f"{DATA_DIR}/maskConnComp.h5"

velocity,  _    = readfile.read(vel_path)
coherence, _    = readfile.read(coh_path)
conn_mask, _    = readfile.read(mask_path)

# Step 1: coherence filter
filtered = np.where(coherence >= 0.70, velocity, np.nan)

# Step 2: connected component filter (conn_mask == 1 means connected)
filtered = np.where(conn_mask == 1, filtered, np.nan)

valid = np.sum(~np.isnan(filtered))
print(f"Pixels remaining after both masks: {valid:,}")

Pixels remaining after both masks: 3,101
